# Module 7 — EA-ATCNet-3C with Grouped Subject Validation and Optional AdaBN

**Purpose:** improve subject-independent 3-class motor-imagery EEG generalization on the frozen BCI-IV-2a cache.

**Input:** `(N, 22, 640)` at 160 Hz.

**Outer evaluation:** strict BCI-IV-2a 9-subject LOSO (`S01` … `S09`).

**Core model:** ATCNet-style temporal + spatial convolution, 5 sliding windows, parallel MHA/TCN encoders, 3-class head.

**New generalization components:**
1. Subject-wise Euclidean alignment (EA) computed from each source subject's training data.
2. Grouped subject validation: complete source subjects are held out for validation.
3. Optional unsupervised target AdaBN using **only unlabeled target EEG**.

**Important reporting guardrail:** `USE_TARGET_ADABN=False` gives a strict untouched-target evaluation. `True` is a separate transductive/unsupervised-adaptation result and must be reported separately.

In [1]:
# ============================================================
# CELL 1 — IMPORTS / DEVICE / REPRODUCIBILITY
# ============================================================

from __future__ import annotations

import os
import gc
import json
import math
import copy
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Seed:", SEED)

Device: cpu
Seed: 42


In [2]:
# ============================================================
# CELL 2 — PROJECT PATH + FROZEN MODULE 5/6 CONTRACT
# ============================================================

PROJECT_ROOT_CANDIDATES = [
    Path("/Users/ashokvarmabevara/Project2"),
    Path.cwd(),
    Path.home() / "Project2",
    Path("/mnt/data/Project2"),
]

PROJECT_ROOT = None

for p in PROJECT_ROOT_CANDIDATES:
    if (
        (p / "cross_dataset_mi_project").exists()
        or (p / "BCI IV-2a").exists()
    ):
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")

PROJECT_DIR = PROJECT_ROOT / "cross_dataset_mi_project"
CACHE_DIR = PROJECT_DIR / "cache"
MANIFEST_DIR = PROJECT_DIR / "manifests"
RESULT_DIR = PROJECT_DIR / "results" / "module_7_ea_atcnet"
FIG_DIR = PROJECT_DIR / "figures" / "module_7_ea_atcnet"
CKPT_DIR = PROJECT_DIR / "checkpoints" / "module_7_ea_atcnet"

for p in [RESULT_DIR, FIG_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

FINAL_CACHE_PATH = CACHE_DIR / (
    "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)
MODULE_5_SPEC_PATH = (
    MANIFEST_DIR / "module_5_v2_preprocessing_specification.json"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CACHE:", FINAL_CACHE_PATH)

assert FINAL_CACHE_PATH.exists(), (
    f"Missing cache: {FINAL_CACHE_PATH}"
)
assert MODULE_5_SPEC_PATH.exists(), (
    f"Missing Module 5 spec: {MODULE_5_SPEC_PATH}"
)

with open(
    MODULE_5_SPEC_PATH,
    "r",
    encoding="utf-8",
) as f:
    m5 = json.load(f)

FROZEN_CHANNELS = [
    str(x) for x in m5["channels"]
]
TARGET_SFREQ = float(
    m5["target_sfreq_hz"]
)
LOW_HZ, HIGH_HZ = map(
    float,
    m5["primary_bandpass_hz"],
)
N_SAMPLES = int(
    m5["epoch"]["n_samples"]
)

CLASSES = [
    "left",
    "right",
    "feet",
]
CLASS_TO_ID = {
    c: i for i, c in enumerate(CLASSES)
}
ID_TO_CLASS = {
    i: c for c, i in CLASS_TO_ID.items()
}
N_CLASSES = len(CLASSES)

assert len(FROZEN_CHANNELS) == 22
assert TARGET_SFREQ == 160.0
assert (LOW_HZ, HIGH_HZ) == (8.0, 30.0)
assert N_SAMPLES == 640
assert N_CLASSES == 3

print({
    "channels": len(FROZEN_CHANNELS),
    "sampling_rate_hz": TARGET_SFREQ,
    "bandpass_hz": (LOW_HZ, HIGH_HZ),
    "samples": N_SAMPLES,
    "classes": CLASSES,
})

PROJECT_ROOT: /Users/ashokvarmabevara/Project2
CACHE: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5
{'channels': 22, 'sampling_rate_hz': 160.0, 'bandpass_hz': (8.0, 30.0), 'samples': 640, 'classes': ['left', 'right', 'feet']}


In [3]:
# ============================================================
# CELL 3 — LOAD HDF5 CACHE + BCI-IV-2a METADATA
# ============================================================

def decode(v):
    if isinstance(v, bytes):
        return v.decode("utf-8")
    return str(v)

with h5py.File(
    FINAL_CACHE_PATH,
    "r",
) as h5:

    X_shape = tuple(
        h5["X"].shape
    )
    X_dtype = str(
        h5["X"].dtype
    )

    metadata = {}

    for k in [
        "dataset",
        "subject",
        "run",
        "recording_id",
        "filename",
        "absolute_path",
        "harmonized_class",
    ]:
        metadata[k] = [
            decode(v)
            for v in h5["metadata"][k][:]
        ]

    metadata["event_index"] = (
        h5["metadata"]["event_index"][:]
        .astype(np.int64)
    )

    metadata["onset_sec"] = (
        h5["metadata"]["onset_sec"][:]
        .astype(np.float64)
    )

    metadata["source_sfreq_hz"] = (
        h5["metadata"]["source_sfreq_hz"][:]
        .astype(np.float64)
    )

cache_meta_df = pd.DataFrame(
    metadata
)

cache_meta_df.insert(
    0,
    "cache_index",
    np.arange(
        len(cache_meta_df),
        dtype=np.int64,
    ),
)

assert X_shape[1:] == (22, 640)
assert X_dtype == "float32"
assert set(
    cache_meta_df["harmonized_class"].unique()
) == set(CLASSES)

bci_meta = cache_meta_df[
    cache_meta_df["dataset"].astype(str)
    == "BCI-IV-2a"
].copy()

bci_meta["subject"] = (
    bci_meta["subject"]
    .astype(str)
)

BCI_SUBJECTS = sorted(
    bci_meta["subject"].unique()
)

assert len(BCI_SUBJECTS) == 9

print(
    "Cache shape:",
    X_shape,
)
print(
    "BCI subjects:",
    BCI_SUBJECTS,
)
print(
    "BCI epochs:",
    len(bci_meta),
)

display(
    bci_meta.groupby("subject")[
        "harmonized_class"
    ]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(
        index=BCI_SUBJECTS,
        columns=CLASSES,
        fill_value=0,
    )
)

Cache shape: (9316, 22, 640)
BCI subjects: ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09']
BCI epochs: 1944


harmonized_class,left,right,feet
subject,,,
S01,72,72,72
S02,72,72,72
S03,72,72,72
S04,72,72,72
S05,72,72,72
S06,72,72,72
S07,72,72,72
S08,72,72,72
S09,72,72,72


In [4]:
# ============================================================
# CELL 4 — HDF5 RANDOM ACCESS + QA
# ============================================================

class HDF5EpochStore:
    def __init__(self, path):
        self.path = Path(path)
        self._h5 = None

    def __enter__(self):
        self._h5 = h5py.File(
            self.path,
            "r",
        )
        return self

    def __exit__(
        self,
        exc_type,
        exc,
        tb,
    ):
        if self._h5 is not None:
            self._h5.close()
            self._h5 = None

    def get_X(
        self,
        indices,
    ):
        idx = np.asarray(
            indices,
            dtype=np.int64,
        )
        values = np.asarray(
            self._h5["X"][idx],
            dtype=np.float32,
        )
        return values


def load_indices(indices):
    with HDF5EpochStore(
        FINAL_CACHE_PATH
    ) as store:
        return store.get_X(
            indices
        )

qa_idx = np.arange(
    min(512, X_shape[0]),
    dtype=np.int64,
)

X_qa = load_indices(
    qa_idx
)

print(
    "QA shape:",
    X_qa.shape,
)

print(
    "Non-finite:",
    int(
        (~np.isfinite(X_qa)).sum()
    ),
)

print(
    "Zero-variance:",
    int(
        (
            np.var(
                X_qa,
                axis=(1, 2),
            )
            <= 1e-12
        ).sum()
    ),
)

assert np.isfinite(
    X_qa
).all()

print(
    "✅ QA PASS"
)

QA shape: (512, 22, 640)
Non-finite: 0
Zero-variance: 0
✅ QA PASS


In [27]:
# ============================================================
# CELL 5 — STRATIFIED SOURCE TRIAL VALIDATION
# ============================================================

def split_source_trials(
    indices,
    y,
    seed=SEED,
    val_fraction=0.20,
):
    """
    Stratified trial-level validation.

    The outer target subject remains completely excluded.

    This is the validation strategy used by the previous
    high-performing ATCNet experiment.
    """

    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    if len(indices) != len(y):
        raise ValueError(
            "indices/y length mismatch."
        )

    rng = np.random.default_rng(
        seed
    )

    train_parts = []
    val_parts = []

    for cls in range(
        N_CLASSES
    ):

        cls_idx = indices[
            y == cls
        ].copy()

        rng.shuffle(
            cls_idx
        )

        n_val = max(
            1,
            int(
                round(
                    len(cls_idx)
                    * val_fraction
                )
            ),
        )

        val_parts.append(
            cls_idx[
                :n_val
            ]
        )

        train_parts.append(
            cls_idx[
                n_val:
            ]
        )

    train_idx = np.concatenate(
        train_parts
    )

    val_idx = np.concatenate(
        val_parts
    )

    rng.shuffle(
        train_idx
    )

    rng.shuffle(
        val_idx
    )

    return (
        train_idx,
        val_idx,
    )


print(
    "✅ Stratified trial split ready."
)

✅ Stratified trial split ready.


In [28]:
# ============================================================
# CELL 6 — SOURCE-ONLY ROBUST NORMALIZATION
# ============================================================

class SourceOnlyRobustNormalizer:

    def __init__(
        self,
        eps=1e-6,
    ):

        self.eps = eps
        self.median_ = None
        self.iqr_ = None

    def fit(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        if X.ndim != 3:
            raise ValueError(
                f"Expected X=(N,C,T), got {X.shape}"
            )

        values = (
            X.transpose(
                1,
                0,
                2,
            )
            .reshape(
                X.shape[1],
                -1,
            )
            .astype(
                np.float64
            )
        )

        self.median_ = (
            np.median(
                values,
                axis=1,
            )
        )

        q25 = np.percentile(
            values,
            25,
            axis=1,
        )

        q75 = np.percentile(
            values,
            75,
            axis=1,
        )

        self.iqr_ = np.maximum(
            q75 - q25,
            self.eps,
        )

        return self

    def transform(
        self,
        X,
    ):

        X = np.asarray(
            X,
            dtype=np.float32,
        )

        if self.median_ is None:
            raise RuntimeError(
                "Normalizer has not been fitted."
            )

        Z = (
            X
            - self.median_[
                None,
                :,
                None,
            ]
        ) / (
            self.iqr_[
                None,
                :,
                None,
            ]
            + self.eps
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z.astype(
            np.float32
        )


print(
    "✅ Source-only robust normalizer ready."
)

✅ Source-only robust normalizer ready.


In [29]:
# ============================================================
# CELL 7 — OPTIONAL TARGET EA POLICY
# ============================================================

# ------------------------------------------------------------
# STRICT MODE:
#   Target subject is NOT used to estimate EA.
#
# TRANSDUCTIVE MODE:
#   Unlabeled target EEG is used to estimate target EA.
#   Target labels are still never used.
#
# Report these as separate experiments.
# ------------------------------------------------------------

USE_TARGET_EA = True

print(
    "USE_TARGET_EA:",
    USE_TARGET_EA,
)

if USE_TARGET_EA:
    print(
        "Mode: TRANSDUCTIVE UNSUPERVISED TARGET EA"
    )
else:
    print(
        "Mode: STRICT UNSEEN-TARGET EA"
    )

USE_TARGET_EA: True
Mode: TRANSDUCTIVE UNSUPERVISED TARGET EA


In [30]:
# ============================================================
# CELL 8 — ATCNET TCN BLOCK
# ============================================================

class CausalConv1d(
    nn.Module
):

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel_size,
        dilation=1,
        bias=True,
    ):

        super().__init__()

        self.pad = (
            kernel_size - 1
        ) * dilation

        self.conv = nn.Conv1d(
            in_ch,
            out_ch,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=self.pad,
            bias=bias,
        )

    def forward(
        self,
        x,
    ):

        y = self.conv(x)

        if self.pad > 0:
            y = y[..., :-self.pad]

        return y


class TCNResidualBlock(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        filters=32,
        depth=2,
        kernel_size=4,
        dropout=0.30,
    ):

        super().__init__()

        self.proj = (
            nn.Conv1d(
                input_dim,
                filters,
                1,
            )
            if input_dim != filters
            else nn.Identity()
        )

        self.blocks = nn.ModuleList()

        for i in range(
            depth
        ):

            dilation = (
                2 ** i
            )

            self.blocks.append(
                nn.ModuleDict(
                    {
                        "c1":
                            CausalConv1d(
                                filters,
                                filters,
                                kernel_size,
                                dilation=dilation,
                            ),

                        "bn1":
                            nn.BatchNorm1d(
                                filters
                            ),

                        "c2":
                            CausalConv1d(
                                filters,
                                filters,
                                kernel_size,
                                dilation=dilation,
                            ),

                        "bn2":
                            nn.BatchNorm1d(
                                filters
                            ),

                        "drop":
                            nn.Dropout(
                                dropout
                            ),
                    }
                )
            )

    def forward(
        self,
        x,
    ):
        # x = (B, L, C)
        z = x.transpose(
            1,
            2,
        )

        residual = self.proj(
            z
        )

        for block in self.blocks:

            h = block["c1"](
                residual
            )

            h = block["bn1"](
                h
            )

            h = F.elu(
                h
            )

            h = block["drop"](
                h
            )

            h = block["c2"](
                h
            )

            h = block["bn2"](
                h
            )

            h = F.elu(
                h
            )

            h = block["drop"](
                h
            )

            residual = F.elu(
                h + residual
            )

        return residual.transpose(
            1,
            2,
        )


print(
    "✅ TCN block ready."
)

✅ TCN block ready.


In [31]:
# ============================================================
# CELL 9 — ATCNET CONVOLUTIONAL BLOCK
# ============================================================

class ATCNetConvBlock(
    nn.Module
):

    def __init__(
        self,
        n_channels=22,
        F1=16,
        D=2,
        kernel_size=64,
        pool1=8,
        pool2=7,
        dropout=0.30,
    ):

        super().__init__()

        F2 = F1 * D

        self.F2 = F2

        self.temporal = nn.Conv2d(
            1,
            F1,
            kernel_size=(
                kernel_size,
                1,
            ),
            padding=(
                kernel_size // 2,
                0,
            ),
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            F1
        )

        self.spatial = nn.Conv2d(
            F1,
            F2,
            kernel_size=(
                1,
                n_channels,
            ),
            groups=F1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            F2
        )

        self.pool1 = nn.AvgPool2d(
            (
                pool1,
                1,
            )
        )

        self.drop1 = nn.Dropout(
            dropout
        )

        self.refine = nn.Conv2d(
            F2,
            F2,
            kernel_size=(
                16,
                1,
            ),
            padding=(
                8,
                0,
            ),
            bias=False,
        )

        self.bn3 = nn.BatchNorm2d(
            F2
        )

        self.pool2 = nn.AvgPool2d(
            (
                pool2,
                1,
            )
        )

        self.drop2 = nn.Dropout(
            dropout
        )

    def forward(
        self,
        x,
    ):

        x = x.unsqueeze(
            1
        ).transpose(
            2,
            3,
        )

        z = self.temporal(
            x
        )

        z = self.bn1(
            z
        )

        z = self.spatial(
            z
        )

        z = self.bn2(
            z
        )

        z = F.elu(
            z
        )

        z = self.pool1(
            z
        )

        z = self.drop1(
            z
        )

        z = self.refine(
            z
        )

        z = self.bn3(
            z
        )

        z = F.elu(
            z
        )

        z = self.pool2(
            z
        )

        z = self.drop2(
            z
        )

        z = z.squeeze(
            -1
        )

        z = z.transpose(
            1,
            2,
        )

        return z


print(
    "✅ ATCNet convolutional block ready."
)

✅ ATCNet convolutional block ready.


In [32]:
# ============================================================
# CELL 10 — ATCNET-3C MODEL
# ============================================================

class ATCNet3C(
    nn.Module
):

    def __init__(
        self,
        n_classes=3,
        n_channels=22,
        n_samples=640,
        n_windows=5,
        F1=16,
        D=2,
        kernel_size=64,
        pool1=8,
        pool2=7,
        attn_heads=2,
        attn_dropout=0.30,
        tcn_depth=2,
        tcn_kernel=4,
        tcn_filters=32,
        tcn_dropout=0.30,
        dropout=0.30,
    ):

        super().__init__()

        self.n_windows = (
            n_windows
        )

        self.n_classes = (
            n_classes
        )

        self.F2 = (
            F1 * D
        )

        self.conv_block = (
            ATCNetConvBlock(
                n_channels=n_channels,
                F1=F1,
                D=D,
                kernel_size=kernel_size,
                pool1=pool1,
                pool2=pool2,
                dropout=dropout,
            )
        )

        self.attention_blocks = (
            nn.ModuleList(
                [
                    nn.MultiheadAttention(
                        embed_dim=self.F2,
                        num_heads=attn_heads,
                        dropout=attn_dropout,
                        batch_first=True,
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        self.attn_norms = (
            nn.ModuleList(
                [
                    nn.LayerNorm(
                        self.F2
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        self.tcn_blocks = (
            nn.ModuleList(
                [
                    TCNResidualBlock(
                        input_dim=self.F2,
                        filters=tcn_filters,
                        depth=tcn_depth,
                        kernel_size=tcn_kernel,
                        dropout=tcn_dropout,
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        self.window_heads = (
            nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Linear(
                            tcn_filters,
                            64,
                        ),
                        nn.ELU(),
                        nn.Dropout(
                            0.25
                        ),
                        nn.Linear(
                            64,
                            n_classes,
                        ),
                    )
                    for _ in range(
                        n_windows
                    )
                ]
            )
        )

        with torch.no_grad():

            dummy = torch.zeros(
                2,
                n_channels,
                n_samples,
            )

            seq = self.conv_block(
                dummy
            )

            self.condensed_length = (
                seq.shape[1]
            )

            self.window_length = (
                self.condensed_length
                - n_windows
                + 1
            )

    def forward(
        self,
        x,
    ):

        seq = self.conv_block(
            x
        )

        B, Tc, C = seq.shape

        if self.window_length <= 0:
            raise RuntimeError(
                "Invalid sliding-window geometry."
            )

        logits = []

        for i in range(
            self.n_windows
        ):

            start = i

            end = (
                Tc
                - self.n_windows
                + i
                + 1
            )

            w = seq[
                :,
                start:end,
                :,
            ]

            attn_out, _ = (
                self.attention_blocks[i](
                    w,
                    w,
                    w,
                    need_weights=False,
                )
            )

            w = self.attn_norms[i](
                w + attn_out
            )

            w = self.tcn_blocks[i](
                w
            )

            last = w[
                :,
                -1,
                :,
            ]

            logits.append(
                self.window_heads[i](
                    last
                )
            )

        return torch.stack(
            logits,
            dim=0,
        ).mean(
            dim=0
        )


# ------------------------------------------------------------
# Shape test
# ------------------------------------------------------------

_test_model = ATCNet3C().to(
    device
)

_params = sum(
    p.numel()
    for p in _test_model.parameters()
    if p.requires_grad
)

with torch.no_grad():

    _dummy = torch.randn(
        2,
        22,
        640,
        device=device,
    )

    _out = _test_model(
        _dummy
    )

print(
    "Condensed length:",
    _test_model.condensed_length,
)

print(
    "Window length:",
    _test_model.window_length,
)

print(
    "Trainable parameters:",
    f"{_params:,}",
)

print(
    "Output shape:",
    tuple(_out.shape)
)

assert tuple(
    _out.shape
) == (
    2,
    3,
)

print(
    "✅ ATCNet-3C forward test PASS"
)

Condensed length: 11
Window length: 7
Trainable parameters: 135,087
Output shape: (2, 3)
✅ ATCNet-3C forward test PASS


In [33]:
# ============================================================
# CELL 11 — SUBJECT-PRESERVING AUGMENTATION + LOADERS
# ============================================================

def augment_mi_eeg(
    x,
):
    """
    Mild augmentation only.
    """

    x = x.clone()

    B, C, T = x.shape

    # Amplitude scaling
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.35:

        x = (
            x
            * torch.empty(
                B,
                1,
                1,
                device=x.device,
            ).uniform_(
                0.92,
                1.08,
            )
        )

    # Very small sensor noise
    if torch.rand(
        1,
        device=x.device,
    ).item() < 0.20:

        x = (
            x
            + 0.003
            * torch.randn_like(
                x
            )
        )

    return x


def make_balanced_loader(
    X,
    y,
    batch_size=64,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    y = np.asarray(
        y,
        dtype=np.int64,
    )

    ds = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(y),
    )

    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(
        np.float64
    )

    inv = np.zeros_like(
        counts
    )

    valid = counts > 0

    inv[valid] = (
        1.0
        / counts[valid]
    )

    weights = inv[y]

    sampler = (
        WeightedRandomSampler(
            torch.as_tensor(
                weights,
                dtype=torch.double,
            ),
            num_samples=len(y),
            replacement=True,
        )
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=False,
    )


def make_eval_loader(
    X,
    batch_size=128,
):

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    ds = TensorDataset(
        torch.from_numpy(X),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
    )


print(
    "✅ Data loaders ready."
)

✅ Data loaders ready.


In [34]:
# ============================================================
# CELL 12 — INFERENCE + TRAINING
# ============================================================

@torch.no_grad()
def predict_model(
    model,
    X,
):
    """
    Return probabilities.
    """

    model.eval()

    loader = make_eval_loader(
        X
    )

    logits_all = []

    for xb, _ in loader:

        xb = xb.to(
            device,
            non_blocking=True,
        )

        logits = model(
            xb
        )

        if isinstance(
            logits,
            tuple,
        ):
            logits = logits[0]

        logits_all.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        logits_all,
        axis=0,
    )

    logits -= logits.max(
        axis=1,
        keepdims=True,
    )

    probs = np.exp(
        logits
    )

    probs /= (
        probs.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return probs.astype(
        np.float32
    )


def fit_class_weights(
    y,
):
    counts = np.bincount(
        y,
        minlength=N_CLASSES,
    ).astype(
        np.float32
    )

    weights = (
        counts.sum()
        /
        (
            N_CLASSES
            * np.maximum(
                counts,
                1.0,
            )
        )
    )

    weights /= (
        weights.mean()
        + 1e-12
    )

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=device,
    )


def train_atcnet(
    X_train,
    y_train,
    X_val,
    y_val,
    seed=SEED,
    epochs=150,
    batch_size=64,
    lr=9e-4,
    patience=25,
):
    """
    Grouped-source-validation training.

    Best checkpoint is selected ONLY on source validation.
    """

    seed_everything(
        seed
    )

    model = ATCNet3C(
        n_classes=3,
        n_channels=22,
        n_samples=640,
        n_windows=5,
        F1=16,
        D=2,
        kernel_size=64,
        pool1=8,
        pool2=7,
        attn_heads=2,
        attn_dropout=0.30,
        tcn_depth=2,
        tcn_kernel=4,
        tcn_filters=32,
        tcn_dropout=0.30,
        dropout=0.30,
    ).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=fit_class_weights(
            y_train
        ),
        label_smoothing=0.01,
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=2e-4,
        betas=(
            0.9,
            0.999,
        ),
    )

    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.8,
            patience=8,
            min_lr=1e-5,
        )
    )

    loader = make_balanced_loader(
        X_train,
        y_train,
        batch_size=batch_size,
    )

    best_state = None
    best_loss = np.inf
    best_bacc = -np.inf
    best_epoch = 0
    wait = 0

    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        train_true = []
        train_pred = []
        batch_losses = []

        for xb, yb in loader:

            xb = xb.to(
                device,
                non_blocking=True,
            )

            yb = yb.to(
                device,
                non_blocking=True,
            )

            xb_aug = (
                augment_mi_eeg(
                    xb
                )
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                xb_aug
            )

            if isinstance(
                logits,
                tuple,
            ):
                logits = logits[0]

            loss = criterion(
                logits,
                yb,
            )

            if not torch.isfinite(
                loss
            ):
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            batch_losses.append(
                float(
                    loss.item()
                )
            )

            train_true.extend(
                yb.detach()
                .cpu()
                .numpy()
            )

            train_pred.extend(
                logits.argmax(
                    1
                )
                .detach()
                .cpu()
                .numpy()
            )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        P_val = predict_model(
            model,
            X_val,
        )

        val_pred = (
            P_val.argmax(
                axis=1
            )
        )

        train_acc = (
            accuracy_score(
                train_true,
                train_pred,
            )
            * 100.0
        )

        val_acc = (
            accuracy_score(
                y_val,
                val_pred,
            )
            * 100.0
        )

        val_bacc = (
            balanced_accuracy_score(
                y_val,
                val_pred,
            )
            * 100.0
        )

        val_loss = -float(
            np.mean(
                np.log(
                    np.clip(
                        P_val[
                            np.arange(
                                len(y_val)
                            ),
                            y_val,
                        ],
                        1e-8,
                        1.0,
                    )
                )
            )
        )

        scheduler.step(
            val_loss
        )

        history.append(
            {
                "epoch":
                    epoch,
                "loss":
                    float(
                        np.mean(
                            batch_losses
                        )
                    ),
                "val_loss":
                    val_loss,
                "train_acc":
                    train_acc,
                "val_acc":
                    val_acc,
                "val_bacc":
                    val_bacc,
                "lr":
                    optimizer.param_groups[
                        0
                    ]["lr"],
            }
        )

        # ----------------------------------------------------
        # Use validation LOSS for checkpointing.
        # ----------------------------------------------------

        if val_loss < (
            best_loss - 1e-5
        ):

            best_loss = val_loss
            best_bacc = val_bacc
            best_epoch = epoch
            wait = 0

            best_state = copy.deepcopy(
                model.state_dict()
            )

        else:

            wait += 1

        if (
            epoch == 1
            or epoch % 10 == 0
        ):

            print(
                f"    epoch {epoch:03d} | "
                f"train={train_acc:5.1f}% | "
                f"val={val_acc:5.1f}% | "
                f"bAcc={val_bacc:5.1f}% | "
                f"vLoss={val_loss:.4f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

        if wait >= patience:

            print(
                f"    early stop at "
                f"epoch {epoch}; "
                f"best={best_epoch}"
            )

            break

    if best_state is None:
        raise RuntimeError(
            "No valid checkpoint was produced."
        )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
        best_epoch,
        best_loss,
        best_bacc,
    )


print(
    "✅ ATCNet training utilities ready."
)

✅ ATCNet training utilities ready.


In [35]:
# ============================================================
# CELL 13 — FINAL PRACTICAL ATCNET LOSO FOLD
# ============================================================
#
# Strategy:
#
#   Outer LOSO:
#       target subject completely excluded from training.
#
#   Source validation:
#       stratified 80/20 trial split.
#
#   Normalization:
#       source-only robust normalization.
#
#   Alignment:
#       OFF for source training.
#
#   Ensemble:
#       2 independent seeds.
#
#   Target adaptation:
#       optional AdaBN only after training.
#
# ============================================================

FINAL_USE_ADABN = True

FINAL_SEEDS = (
    42,
    123,
)


def train_final_atcnet_fold(
    target_subject,
    fold_id,
    epochs=180,
    batch_size=64,
    lr=9e-4,
    patience=35,
):

    t0 = time.time()

    target_subject = str(
        target_subject
    )

    # ========================================================
    # METADATA
    # ========================================================

    bci = cache_meta_df[
        cache_meta_df[
            "dataset"
        ].astype(str)
        == "BCI-IV-2a"
    ].copy()

    bci["subject"] = (
        bci["subject"]
        .astype(str)
    )

    source_mask = (
        bci["subject"]
        != target_subject
    )

    target_mask = (
        bci["subject"]
        == target_subject
    )

    source_indices = (
        bci.loc[
            source_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    target_indices = (
        bci.loc[
            target_mask,
            "cache_index",
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    X_source_raw = load_indices(
        source_indices
    )

    X_target_raw = load_indices(
        target_indices
    )

    y_source = (
        bci.loc[
            source_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    y_target = (
        bci.loc[
            target_mask,
            "harmonized_class",
        ]
        .map(
            CLASS_TO_ID
        )
        .to_numpy(
            dtype=np.int64
        )
    )

    # ========================================================
    # SOURCE-ONLY NORMALIZATION
    # ========================================================

    normalizer = (
        SourceOnlyRobustNormalizer()
        .fit(
            X_source_raw
        )
    )

    X_source = (
        normalizer.transform(
            X_source_raw
        )
    )

    X_target = (
        normalizer.transform(
            X_target_raw
        )
    )

    # ========================================================
    # STRATIFIED SOURCE TRIAL SPLIT
    # ========================================================

    all_source_local = np.arange(
        len(
            X_source
        ),
        dtype=np.int64,
    )

    train_local, val_local = (
        split_source_trials(
            all_source_local,
            y_source,
            seed=SEED,
            val_fraction=0.20,
        )
    )

    X_train = (
        X_source[
            train_local
        ]
    )

    y_train = (
        y_source[
            train_local
        ]
    )

    X_val = (
        X_source[
            val_local
        ]
    )

    y_val = (
        y_source[
            val_local
        ]
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        f"FINAL ATCNet LOSO "
        f"[{fold_id}/9] — target {target_subject}"
    )

    print(
        "=" * 78
    )

    print(
        "Source trials:",
        len(X_source)
    )

    print(
        "Train trials :",
        len(X_train)
    )

    print(
        "Val trials   :",
        len(X_val)
    )

    print(
        "Test trials  :",
        len(X_target)
    )

    print(
        "Seeds        :",
        FINAL_SEEDS
    )

    print(
        "AdaBN        :",
        FINAL_USE_ADABN
    )

    # ========================================================
    # TRAIN TWO MODELS
    # ========================================================

    models = []

    seed_info = []

    for seed in FINAL_SEEDS:

        model, hist, best_epoch, best_loss, best_bacc = (
            train_atcnet(
                X_train,
                y_train,
                X_val,
                y_val,
                seed=seed,
                epochs=epochs,
                batch_size=batch_size,
                lr=lr,
                patience=patience,
            )
        )

        models.append(
            model
        )

        seed_info.append(
            {
                "seed":
                    seed,
                "best_epoch":
                    best_epoch,
                "best_val_loss":
                    best_loss,
                "best_val_bacc":
                    best_bacc,
            }
        )

    seed_df = pd.DataFrame(
        seed_info
    )

    print(
        "\nSeed summary:"
    )

    display(
        seed_df
    )

    # ========================================================
    # STANDARD ENSEMBLE
    # ========================================================

    P_test_list = []

    for model in models:

        P = model_predict(
            model,
            X_target,
        )

        P_test_list.append(
            P
        )

    P_standard = np.mean(
        np.stack(
            P_test_list,
            axis=0,
        ),
        axis=0,
    )

    pred_standard = (
        P_standard.argmax(
            axis=1
        )
    )

    standard_acc = (
        accuracy_score(
            y_target,
            pred_standard,
        )
        * 100.0
    )

    standard_bacc = (
        balanced_accuracy_score(
            y_target,
            pred_standard,
        )
        * 100.0
    )

    # ========================================================
    # OPTIONAL UNSUPERVISED TARGET ADABN
    # ========================================================

    if FINAL_USE_ADABN:

        adapted_models = []

        for model in models:

            adapted = (
                update_batchnorm_from_unlabeled_target(
                    model,
                    X_target,
                    batch_size=64,
                )
            )

            adapted_models.append(
                adapted
            )

        P_adapted = []

        for model in adapted_models:

            P = model_predict(
                model,
                X_target,
            )

            P_adapted.append(
                P
            )

        P_adabn = np.mean(
            np.stack(
                P_adapted,
                axis=0,
            ),
            axis=0,
        )

        pred_adabn = (
            P_adabn.argmax(
                axis=1
            )
        )

        adabn_acc = (
            accuracy_score(
                y_target,
                pred_adabn,
            )
            * 100.0
        )

        adabn_bacc = (
            balanced_accuracy_score(
                y_target,
                pred_adabn,
            )
            * 100.0
        )

        final_P = P_adabn
        final_pred = pred_adabn
        final_acc = adabn_acc
        final_bacc = adabn_bacc

    else:

        adabn_acc = np.nan
        adabn_bacc = np.nan

        final_P = P_standard
        final_pred = pred_standard
        final_acc = standard_acc
        final_bacc = standard_bacc

    kappa = cohen_kappa_score(
        y_target,
        final_pred,
    )

    elapsed = (
        time.time()
        - t0
    )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Target subject        : "
        f"{target_subject}"
    )

    print(
        f"Validation bAcc       : "
        f"{seed_df['best_val_bacc'].mean():.2f}%"
    )

    print(
        f"Standard ensemble     : "
        f"{standard_acc:.2f}%"
    )

    print(
        f"Standard bAcc         : "
        f"{standard_bacc:.2f}%"
    )

    if FINAL_USE_ADABN:

        print(
            f"AdaBN ensemble        : "
            f"{adabn_acc:.2f}%"
        )

        print(
            f"AdaBN bAcc            : "
            f"{adabn_bacc:.2f}%"
        )

    print(
        f"FINAL accuracy        : "
        f"{final_acc:.2f}%"
    )

    print(
        f"FINAL bAcc            : "
        f"{final_bacc:.2f}%"
    )

    print(
        f"Kappa                 : "
        f"{kappa:.4f}"
    )

    print(
        f"Elapsed               : "
        f"{elapsed / 60.0:.1f} min"
    )

    print(
        "-" * 78
    )

    return {
        "fold":
            fold_id,

        "subject":
            target_subject,

        "accuracy":
            final_acc,

        "bacc":
            final_bacc,

        "standard_acc":
            standard_acc,

        "standard_bacc":
            standard_bacc,

        "adabn_acc":
            adabn_acc,

        "adabn_bacc":
            adabn_bacc,

        "kappa":
            kappa,

        "val_bacc":
            float(
                seed_df[
                    "best_val_bacc"
                ].mean()
            ),

        "models":
            models,

        "seed_summary":
            seed_df,

        "y_test":
            y_target,

        "pred_test":
            final_pred,

        "P_test":
            final_P,
    }


print(
    "✅ Final ATCNet LOSO fold ready."
)

✅ Final ATCNet LOSO fold ready.


In [36]:
# ============================================================
# CELL 14 — OPTIONAL S01 SMOKE TEST
# ============================================================

RUN_SMOKE = True

if RUN_SMOKE:

    smoke_result = (
        run_ea_atcnet_fold(
            target_subject="S01",
            fold_id=1,
            seed=SEED,
            epochs=150,
            batch_size=64,
            lr=9e-4,
            patience=25,
            n_seeds=2,
        )
    )

    print(
        "\nS01 smoke accuracy:",
        f"{smoke_result['test_acc']:.2f}%"
    )

    print(
        "S01 smoke bAcc:",
        f"{smoke_result['test_bacc']:.2f}%"
    )


EA-ATCNet LOSO [1/9] — target S01
Source subjects: 8
Train subjects: ['S02', 'S03', 'S04', 'S07', 'S08', 'S09']
Val subjects  : ['S05', 'S06']
Target subject: S01
Target EA mode: transductive
Train trials: 1296
Val trials: 432
Test trials: 216
    epoch 001 | train= 38.1% | val= 33.3% | bAcc= 33.3% | vLoss=1.1642 | lr=9.00e-04
    epoch 010 | train= 65.7% | val= 38.4% | bAcc= 38.4% | vLoss=1.5641 | lr=9.00e-04
    epoch 020 | train= 70.1% | val= 38.2% | bAcc= 38.2% | vLoss=2.0616 | lr=5.76e-04
    early stop at epoch 27; best=2
    epoch 001 | train= 37.6% | val= 33.3% | bAcc= 33.3% | vLoss=1.1224 | lr=9.00e-04
    epoch 010 | train= 69.1% | val= 38.0% | bAcc= 38.0% | vLoss=1.6412 | lr=7.20e-04
    epoch 020 | train= 72.0% | val= 37.0% | bAcc= 37.0% | vLoss=2.1841 | lr=5.76e-04
    early stop at epoch 26; best=1


,seed,best_epoch,best_val_loss,best_val_bacc
0,42,2,1.106460,33.333333
1,1051,1,1.122358,33.333333



------------------------------------------------------------------------------
Target subject        : S01
Grouped validation    : 33.33%
External test accuracy: 33.33%
External test bAcc    : 33.33%
Cohen kappa           : 0.0000
Elapsed               : 5.2 min
------------------------------------------------------------------------------

S01 smoke accuracy: 33.33%
S01 smoke bAcc: 33.33%


In [38]:
# ============================================================
# CELL 15 — S01 FINAL SMOKE TEST
# FIXED: DEFINES model_predict() BEFORE EVALUATION
# ============================================================

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
)


# ============================================================
# 1. SAFE MODEL PREDICTOR
# ============================================================

@torch.no_grad()
def model_predict(
    model,
    X,
    batch_size=128,
):
    """
    Predict probabilities from ATCNet.

    Returns:
        P : (N, 3)
    """

    model.eval()

    X = np.asarray(
        X,
        dtype=np.float32,
    )

    if X.ndim != 3:
        raise ValueError(
            f"Expected X=(N,C,T), got {X.shape}"
        )

    if X.shape[1:] != (
        22,
        640,
    ):
        raise ValueError(
            f"Expected (N,22,640), got {X.shape}"
        )

    dataset = TensorDataset(
        torch.from_numpy(X),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    logits_all = []

    for xb, _ in loader:

        xb = xb.to(
            device,
            non_blocking=True,
        )

        output = model(
            xb
        )

        if isinstance(
            output,
            tuple,
        ):

            logits = output[0]

        else:

            logits = output

        if not torch.isfinite(
            logits
        ).all():

            raise RuntimeError(
                "Non-finite logits encountered."
            )

        logits_all.append(
            logits.detach()
            .cpu()
            .numpy()
        )

    logits = np.concatenate(
        logits_all,
        axis=0,
    )

    # Stable softmax.
    logits = (
        logits
        - logits.max(
            axis=1,
            keepdims=True,
        )
    )

    P = np.exp(
        logits
    )

    P /= (
        P.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    return P.astype(
        np.float32
    )


# ============================================================
# 2. CHECK THAT THE TRAINED MODELS EXIST
# ============================================================

if "models" not in globals():

    raise RuntimeError(
        "Variable 'models' does not exist.\n"
        "The training cell must finish successfully first."
    )

if len(models) == 0:

    raise RuntimeError(
        "The models list is empty."
    )

print(
    "Trained models:",
    len(models)
)


# ============================================================
# 3. VERIFY TARGET TEST DATA
# ============================================================

if "X_target" not in globals():

    raise RuntimeError(
        "X_target does not exist."
    )

if "y_target" not in globals():

    raise RuntimeError(
        "y_target does not exist."
    )

print(
    "Target EEG:",
    X_target.shape
)

print(
    "Target labels:",
    y_target.shape
)


# ============================================================
# 4. STANDARD MULTI-SEED ENSEMBLE
# ============================================================

probabilities = []

for i, model in enumerate(
    models
):

    print(
        f"Predicting with seed model "
        f"{i + 1}/{len(models)}..."
    )

    P = model_predict(
        model,
        X_target,
    )

    probabilities.append(
        P
    )


P_standard = np.mean(
    np.stack(
        probabilities,
        axis=0,
    ),
    axis=0,
)

P_standard /= (
    P_standard.sum(
        axis=1,
        keepdims=True,
    )
    + 1e-12
)

pred_standard = (
    P_standard.argmax(
        axis=1
    )
)


# ============================================================
# 5. OPTIONAL TARGET AdaBN
# ============================================================

def update_batchnorm_from_unlabeled_target(
    model,
    X,
    batch_size=64,
):
    """
    Transductive AdaBN.

    Uses target EEG only to update BN running statistics.
    No target labels are used.
    """

    adapted = copy.deepcopy(
        model
    ).to(device)

    # --------------------------------------------------------
    # First place the model in train mode so BatchNorm
    # updates running statistics.
    # --------------------------------------------------------

    adapted.train()

    # --------------------------------------------------------
    # Freeze all parameters.
    # We only want running statistics to change.
    # --------------------------------------------------------

    for p in adapted.parameters():

        p.requires_grad = False

    # --------------------------------------------------------
    # Keep Dropout disabled.
    # --------------------------------------------------------

    for module in adapted.modules():

        if isinstance(
            module,
            (
                nn.Dropout,
                nn.Dropout1d,
                nn.Dropout2d,
                nn.Dropout3d,
            ),
        ):

            module.eval()

    dataset = TensorDataset(
        torch.from_numpy(
            np.asarray(
                X,
                dtype=np.float32,
            )
        ),
        torch.zeros(
            len(X),
            dtype=torch.long,
        ),
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    with torch.no_grad():

        for xb, _ in loader:

            xb = xb.to(
                device,
                non_blocking=True,
            )

            _ = adapted(
                xb
            )

    adapted.eval()

    return adapted


if "FINAL_USE_ADABN" not in globals():

    FINAL_USE_ADABN = True


if FINAL_USE_ADABN:

    print(
        "\nApplying unlabeled-target AdaBN..."
    )

    adabn_models = []

    for i, model in enumerate(
        models
    ):

        print(
            f"AdaBN model "
            f"{i + 1}/{len(models)}"
        )

        adabn_models.append(
            update_batchnorm_from_unlabeled_target(
                model,
                X_target,
                batch_size=64,
            )
        )

    adabn_probabilities = []

    for model in adabn_models:

        P = model_predict(
            model,
            X_target,
        )

        adabn_probabilities.append(
            P
        )

    P_adabn = np.mean(
        np.stack(
            adabn_probabilities,
            axis=0,
        ),
        axis=0,
    )

    P_adabn /= (
        P_adabn.sum(
            axis=1,
            keepdims=True,
        )
        + 1e-12
    )

    pred_adabn = (
        P_adabn.argmax(
            axis=1
        )
    )

else:

    P_adabn = None
    pred_adabn = None


# ============================================================
# 6. METRICS
# ============================================================

standard_acc = (
    accuracy_score(
        y_target,
        pred_standard,
    )
    * 100.0
)

standard_bacc = (
    balanced_accuracy_score(
        y_target,
        pred_standard,
    )
    * 100.0
)

standard_kappa = (
    cohen_kappa_score(
        y_target,
        pred_standard,
    )
)


if FINAL_USE_ADABN:

    adabn_acc = (
        accuracy_score(
            y_target,
            pred_adabn,
        )
        * 100.0
    )

    adabn_bacc = (
        balanced_accuracy_score(
            y_target,
            pred_adabn,
        )
        * 100.0
    )

    adabn_kappa = (
        cohen_kappa_score(
            y_target,
            pred_adabn,
        )
    )

else:

    adabn_acc = np.nan
    adabn_bacc = np.nan
    adabn_kappa = np.nan


# ============================================================
# 7. FINAL RESULT
# ============================================================

if (
    FINAL_USE_ADABN
    and np.isfinite(adabn_acc)
):

    final_acc = adabn_acc
    final_bacc = adabn_bacc
    final_kappa = adabn_kappa

    final_pred = pred_adabn

    selected_method = (
        "ATCNet ensemble + AdaBN"
    )

else:

    final_acc = standard_acc
    final_bacc = standard_bacc
    final_kappa = standard_kappa

    final_pred = pred_standard

    selected_method = (
        "ATCNet multi-seed ensemble"
    )


# ============================================================
# 8. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "S01 FINAL SMOKE TEST"
)

print(
    "=" * 78
)

print(
    f"Standard ensemble accuracy : "
    f"{standard_acc:.2f}%"
)

print(
    f"Standard ensemble bAcc     : "
    f"{standard_bacc:.2f}%"
)

print(
    f"Standard ensemble kappa    : "
    f"{standard_kappa:.4f}"
)

if FINAL_USE_ADABN:

    print(
        f"AdaBN ensemble accuracy   : "
        f"{adabn_acc:.2f}%"
    )

    print(
        f"AdaBN ensemble bAcc       : "
        f"{adabn_bacc:.2f}%"
    )

    print(
        f"AdaBN ensemble kappa      : "
        f"{adabn_kappa:.4f}"
    )

print(
    "-" * 78
)

print(
    f"SELECTED METHOD            : "
    f"{selected_method}"
)

print(
    f"FINAL ACCURACY             : "
    f"{final_acc:.2f}%"
)

print(
    f"FINAL BALANCED ACCURACY    : "
    f"{final_bacc:.2f}%"
)

print(
    f"FINAL KAPPA                : "
    f"{final_kappa:.4f}"
)

print(
    "=" * 78
)


# ============================================================
# 9. TARGET CHECK
# ============================================================

if final_acc >= 80.0:

    print(
        f"✅ 80% TARGET REACHED — "
        f"{final_acc:.2f}%"
    )

elif final_acc >= 70.0:

    print(
        f"✅ 70% TARGET REACHED — "
        f"{final_acc:.2f}%"
    )

else:

    print(
        f"❌ TARGET NOT YET REACHED — "
        f"{final_acc:.2f}%"
    )

RuntimeError: Variable 'models' does not exist.
The training cell must finish successfully first.

In [ ]:
# ============================================================
# CELL 16 — FULL 9-SUBJECT FINAL LOSO
# ============================================================

RUN_FULL_FINAL_LOSO = True

if RUN_FULL_FINAL_LOSO:

    final_results = []
    final_objects = {}

    for fold_id, subject in enumerate(
        BCI_SUBJECTS,
        start=1,
    ):

        result = (
            train_final_atcnet_fold(
                target_subject=subject,
                fold_id=fold_id,
                epochs=180,
                batch_size=64,
                lr=9e-4,
                patience=35,
            )
        )

        final_objects[
            subject
        ] = result

        final_results.append(
            {
                "fold":
                    fold_id,

                "subject":
                    subject,

                "standard_accuracy":
                    result[
                        "standard_acc"
                    ],

                "standard_bacc":
                    result[
                        "standard_bacc"
                    ],

                "adabn_accuracy":
                    result[
                        "adabn_acc"
                    ],

                "adabn_bacc":
                    result[
                        "adabn_bacc"
                    ],

                "final_accuracy":
                    result[
                        "accuracy"
                    ],

                "final_bacc":
                    result[
                        "bacc"
                    ],

                "kappa":
                    result[
                        "kappa"
                    ],

                "val_bacc":
                    result[
                        "val_bacc"
                    ],
            }
        )

        gc.collect()

    final_results_df = pd.DataFrame(
        final_results
    )

    print(
        "\n"
        + "=" * 78
    )

    print(
        "FINAL ATCNet-3C LOSO"
    )

    print(
        "=" * 78
    )

    display(
        final_results_df
    )

    print(
        "\n"
        "Standard ensemble mean:",
        f"{final_results_df['standard_accuracy'].mean():.2f}%"
    )

    print(
        "AdaBN mean:",
        f"{final_results_df['adabn_accuracy'].mean():.2f}%"
    )

    print(
        "Final mean accuracy:",
        f"{final_results_df['final_accuracy'].mean():.2f}%"
    )

    print(
        "Final mean bAcc:",
        f"{final_results_df['final_bacc'].mean():.2f}%"
    )

    print(
        "Median:",
        f"{final_results_df['final_accuracy'].median():.2f}%"
    )

    print(
        "Std:",
        f"{final_results_df['final_accuracy'].std():.2f}%"
    )

    print(
        "Subjects >=70%:",
        int(
            (
                final_results_df[
                    "final_accuracy"
                ]
                >= 70.0
            ).sum()
        ),
        "/9",
    )

    print(
        "Subjects >=80%:",
        int(
            (
                final_results_df[
                    "final_accuracy"
                ]
                >= 80.0
            ).sum()
        ),
        "/9",
    )

    mean_final = (
        final_results_df[
            "final_accuracy"
        ].mean()
    )

    print(
        "\n"
        + "=" * 78
    )

    if mean_final >= 80.0:

        print(
            f"✅ 80% TARGET ACHIEVED: "
            f"{mean_final:.2f}%"
        )

    elif mean_final >= 70.0:

        print(
            f"✅ 70% TARGET ACHIEVED: "
            f"{mean_final:.2f}%"
        )

    else:

        print(
            f"❌ TARGET NOT YET ACHIEVED: "
            f"{mean_final:.2f}%"
        )

    print(
        "=" * 78
    )

In [ ]:
# ============================================================
# CELL 17 — CONFUSION MATRIX + CLASSIFICATION REPORT
# ============================================================

if (
    "full_fold_objects" in globals()
    and full_fold_objects
):

    all_true = np.concatenate(
        [
            full_fold_objects[s][
                "y_test"
            ]
            for s in BCI_SUBJECTS
        ]
    )

    all_pred = np.concatenate(
        [
            full_fold_objects[s][
                "pred_test"
            ]
            for s in BCI_SUBJECTS
        ]
    )

    cm = confusion_matrix(
        all_true,
        all_pred,
        labels=list(
            range(N_CLASSES)
        ),
        normalize="true",
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"true_{c}"
            for c in CLASSES
        ],
        columns=[
            f"pred_{c}"
            for c in CLASSES
        ],
    )

    print(
        "Normalized confusion matrix:"
    )

    display(
        cm_df.round(3)
    )

    print(
        "\nClassification report:"
    )

    print(
        classification_report(
            all_true,
            all_pred,
            labels=list(
                range(N_CLASSES)
            ),
            target_names=CLASSES,
            digits=4,
        )
    )

    plt.figure(
        figsize=(6, 5)
    )

    plt.imshow(
        cm,
        interpolation="nearest",
    )

    plt.xticks(
        range(N_CLASSES),
        CLASSES,
    )

    plt.yticks(
        range(N_CLASSES),
        CLASSES,
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        "EA-ATCNet-3C — BCI-IV-2a LOSO"
    )

    for i in range(
        N_CLASSES
    ):

        for j in range(
            N_CLASSES
        ):

            plt.text(
                j,
                i,
                f"{cm[i,j]:.2f}",
                ha="center",
                va="center",
            )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR
        / "ea_atcnet_loso_confusion_matrix.png",
        dpi=180,
    )

    plt.show()

In [ ]:
# ============================================================
# CELL 18 — LEARNING CURVE + SUBJECT SUMMARY
# ============================================================

if (
    "full_fold_objects" in globals()
    and full_fold_objects
):

    first_subject = BCI_SUBJECTS[0]

    histories = (
        full_fold_objects[
            first_subject
        ][
            "histories"
        ]
    )

    plt.figure(
        figsize=(8, 5)
    )

    for i, hist in enumerate(
        histories
    ):

        plt.plot(
            hist["epoch"],
            hist["val_bacc"],
            label=f"seed_{i}",
        )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Validation Balanced Accuracy (%)"
    )

    plt.title(
        f"EA-ATCNet Validation — {first_subject}"
    )

    plt.legend()

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        FIG_DIR
        / f"ea_atcnet_validation_{first_subject}.png",
        dpi=180,
    )

    plt.show()


# ------------------------------------------------------------
# Subject ranking
# ------------------------------------------------------------

if "loso_results_df" in globals():

    print(
        "Best targets:"
    )

    display(
        loso_results_df[
            [
                "subject",
                "ea_test_acc",
                "ea_test_bacc",
                "val_bacc",
            ]
        ]
        .sort_values(
            "ea_test_acc",
            ascending=False,
        )
    )

    print(
        "\nWeakest targets:"
    )

    display(
        loso_results_df[
            [
                "subject",
                "ea_test_acc",
                "ea_test_bacc",
                "val_bacc",
            ]
        ]
        .sort_values(
            "ea_test_acc",
            ascending=True,
        )
        .head(5)
    )

In [ ]:
# ============================================================
# CELL 19 — SAVE RESULTS / PROTOCOL
# ============================================================

if "loso_results_df" in globals():

    results_csv = (
        RESULT_DIR
        / "ea_atcnet_3c_bci_loso_results.csv"
    )

    loso_results_df.to_csv(
        results_csv,
        index=False,
    )

    print(
        "Saved:",
        results_csv
    )


protocol = {
    "model":
        "EA-ATCNet-3C",

    "evaluation":
        "BCI-IV-2a within-dataset subject-independent LOSO",

    "n_subjects":
        9,

    "subjects":
        BCI_SUBJECTS,

    "n_channels":
        22,

    "n_samples":
        640,

    "sampling_rate_hz":
        160.0,

    "classes":
        CLASSES,

    "grouped_source_validation":
        True,

    "validation_fraction_subjects":
        0.25,

    "subject_wise_euclidean_alignment":
        True,

    "use_target_ea":
        bool(
            USE_TARGET_EA
        ),

    "use_target_adabn":
        bool(
            USE_TARGET_ADABN
        ),

    "ensemble_seeds":
        [42, 1051],

    "learning_rate":
        9e-4,

    "batch_size":
        64,

    "epochs":
        150,

    "patience":
        25,
}

protocol_path = (
    RESULT_DIR
    / "ea_atcnet_3c_protocol.json"
)

with open(
    protocol_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )

print(
    "Saved protocol:",
    protocol_path
)

## Interpretation guardrail

The result to report for the strict experiment is the mean of `ea_test_acc` across all 9 held-out subjects.

`USE_TARGET_EA=True` and `USE_TARGET_ADABN=True` are **transductive unsupervised adaptation** variants because they use the unlabeled target EEG distribution. They must not be mixed with the strict untouched-target LOSO score.

A >70% result is not guaranteed by the architecture. The purpose of this notebook is to address the observed failure mode in the previous experiment—high source validation performance but poor unseen-subject performance—using subject-wise covariance alignment and grouped validation.